# Isolation Forest: Experimento 02 (Refinamiento)
## Modelo ex ante por Segmento + Clase UNSPSC, con analisis de sensibilidad

**Fuente de datos:** `V_BASE_OFERTA_ITEM.parquet`  
**Cambios respecto a exp_01:**
- Exclusion de variables de resultado (`FUE_ADJUDICADO`, contrato) del vector de features
- Correccion de `PRODUCTO_DIFIERE` (primeros 8 digitos)
- Ampliacion de `TAMANO_PROVEEDOR` (Microemprendedor=1)
- Multiples umbrales (P1, P2, P3, P5, P10)
- Tres reglas de agregacion (`min`, `mean`, `max`)
- Normalizacion z-score por segmento
- Perfil comparativo con medias, medianas y percentiles
- Analisis de estabilidad con 5 semillas
- Reporte estratificado por segmento

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from pathlib import Path
from scipy import stats
import time
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

DATA_PATH = Path('..') / 'datos' / 'V_BASE_OFERTA_ITEM.parquet'
FIG_DIR = Path('figuras')
RES_DIR = Path('resultados')
FIG_DIR.mkdir(exist_ok=True)
RES_DIR.mkdir(exist_ok=True)

SEMILLA_PRINCIPAL = 42
SEMILLAS_ESTABILIDAD = [42, 123, 456, 789, 2026]
MIN_OBS_CLASE = 500

df = pd.read_parquet(DATA_PATH)
print(f'Registros: {len(df):,}  |  Columnas: {df.shape[1]}')

Registros: 3,021,336  |  Columnas: 70


## 1. Diagnostico inicial

In [2]:
print('=== Nulos por columna (top 20) ===')
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].head(20).to_string())

print(f'\n=== Segmentos UNSPSC unicos: {df["OFERTA_SEGMENTO"].nunique()} ===')
seg_counts = df['OFERTA_SEGMENTO'].value_counts()
print(seg_counts.to_string())

print(f'\n=== Clases UNSPSC unicas: {df["OFERTA_CLASE"].nunique()} ===')
clase_counts = df['OFERTA_CLASE'].value_counts()
clases_viables = clase_counts[clase_counts >= MIN_OBS_CLASE]
clases_no_viables = clase_counts[clase_counts < MIN_OBS_CLASE]
print(f'Clases con >= {MIN_OBS_CLASE} obs: {len(clases_viables):,} (cubren {clases_viables.sum():,} ofertas, {clases_viables.sum()/len(df)*100:.1f}%)')
print(f'Clases con <  {MIN_OBS_CLASE} obs: {len(clases_no_viables):,} (cubren {clases_no_viables.sum():,} ofertas, {clases_no_viables.sum()/len(df)*100:.1f}%)')

=== Nulos por columna (top 20) ===


ID_CONSORCIO                       97.843305
EXCEPCION_CD                       64.013569
CONTRATO_TIPO_MONEDA               14.147351
CONTRATO_TIPO_CAMBIO_CRC           14.135932
CONTRATO_PRECIO_UNITARIO           14.132788
CONTRATO_PRECIO_UNITARIO_CRC       14.132788
CONTRATO_CANTIDAD                  14.132788
PRECIO_STDDEV_ITEM_CRC             12.688162
RATIO_OFERTADO_VS_ESTIMADO          0.859587
RATIO_OFERTADO_VS_MIN_ITEM          0.627934
RATIO_OFERTADO_VS_PROM_ITEM         0.356200
NOMBRE_UNIDAD_COMPRA                0.154236
FECHA_CIERRE_RECEPCION              0.147352
NRO_PROCEDIMIENTO                   0.143149
CARTEL_PRECIO_UNITARIO_CRC          0.143149
CARTEL_MERCADERIA                   0.143149
CARTEL_PRECIO_UNITARIO_ESTIMADO     0.143149
CEDULA_INSTITUCION                  0.143149
NOMBRE_INSTITUCION                  0.143149
MODALIDAD_PROCEDIMIENTO             0.143149

=== Segmentos UNSPSC unicos: 57 ===


OFERTA_SEGMENTO
72    202651
42    190003
40    179359
31    174396
30    174094
44    170968
47    139288
41    123457
27    123213
39    121389
46    109049
43    100188
78     91872
81     79461
53     76154
14     63941
55     56362
26     56318
82     52946
56     50357
50     48234
90     46591
80     43791
73     39452
52     39393
86     38696
25     37509
51     35887
85     32315
24     32042
11     26698
23     25078
60     24432
83     23337
49     22673
15     22495
10     20446
76     20237
70     15924
48     15690
92     14794
45     14664
12     12393
32      6193
21      5306
84      5274
93      3761
20      2754
91      2474
77      2101
22      1757
95      1477
13       938
71       573
54       227
94       194
64        70

=== Clases UNSPSC unicas: 2196 ===
Clases con >= 500 obs: 656 (cubren 2,862,578 ofertas, 94.7%)
Clases con <  500 obs: 1,540 (cubren 158,758 ofertas, 5.3%)


## 2. Feature engineering (corregido)

In [3]:
# Log de precios
df['LOG_PRECIO_OFERTADO'] = np.where(
    df['OFERTA_PRECIO_UNITARIO_CRC'] > 0,
    np.log10(df['OFERTA_PRECIO_UNITARIO_CRC']), np.nan)
df['LOG_PRECIO_ESTIMADO'] = np.where(
    df['CARTEL_PRECIO_UNITARIO_CRC'] > 0,
    np.log10(df['CARTEL_PRECIO_UNITARIO_CRC']), np.nan)
df['LOG_RATIO_VS_ESTIMADO'] = np.where(
    df['RATIO_OFERTADO_VS_ESTIMADO'] > 0,
    np.log10(df['RATIO_OFERTADO_VS_ESTIMADO']), np.nan)

# Fechas y features temporales
for col in ['FECHA_PUBLICACION', 'FECHA_CIERRE_RECEPCION',
            'FECHA_APERTURA', 'FECHA_PRESENTA_OFERTA']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df['DIAS_PARA_CIERRE'] = (
    df['FECHA_CIERRE_RECEPCION'] - df['FECHA_PRESENTA_OFERTA']
).dt.total_seconds() / 86400
df['DIAS_VENTANA_PROCESO'] = (
    df['FECHA_CIERRE_RECEPCION'] - df['FECHA_PUBLICACION']
).dt.total_seconds() / 86400

# Ratio cantidad ofertada vs solicitada
df['RATIO_CANTIDAD_VS_SOLICITADA'] = np.where(
    df['CARTEL_CANTIDAD_SOLICITADA'].isna() | (df['CARTEL_CANTIDAD_SOLICITADA'] == 0),
    np.nan, df['CANTIDAD_OFERTADA'] / df['CARTEL_CANTIDAD_SOLICITADA'])

# CORRECCION: PRODUCTO_DIFIERE compara primeros 8 digitos (nivel mercaderia)
oferta_prod_8 = df['OFERTA_CODIGO_PRODUCTO'].astype(str).str[:8]
cartel_prod_8 = df['CARTEL_CODIGO_PRODUCTO'].astype(str).str[:8]
df['PRODUCTO_DIFIERE'] = (oferta_prod_8 != cartel_prod_8).astype(int)
pct_difiere = df['PRODUCTO_DIFIERE'].mean() * 100
print(f'PRODUCTO_DIFIERE (primeros 8 digitos): {pct_difiere:.1f}% difieren')

# CV del precio del item
df['CV_PRECIO_ITEM'] = np.where(
    df['PRECIO_PROM_ITEM_CRC'].isna() | (df['PRECIO_PROM_ITEM_CRC'] == 0),
    np.nan, df['PRECIO_STDDEV_ITEM_CRC'] / df['PRECIO_PROM_ITEM_CRC'])

# Flag regimen legal (Ley 7494 vs Ley 9986)
df['REGIMEN_LGCP'] = (df['FECHA_PRESENTA_OFERTA'] >= '2022-12-01').astype(int)
print(f'REGIMEN_LGCP: {df["REGIMEN_LGCP"].mean()*100:.1f}% bajo Ley 9986')

print('\nFeatures derivadas creadas:')
nuevas = ['LOG_PRECIO_OFERTADO', 'LOG_PRECIO_ESTIMADO', 'LOG_RATIO_VS_ESTIMADO',
          'DIAS_PARA_CIERRE', 'DIAS_VENTANA_PROCESO',
          'RATIO_CANTIDAD_VS_SOLICITADA', 'PRODUCTO_DIFIERE', 'CV_PRECIO_ITEM',
          'REGIMEN_LGCP']
for f in nuevas:
    nn = df[f].notna().sum()
    print(f'  {f:<35s} {nn:>10,} no-null ({nn/len(df)*100:.1f}%)')

PRODUCTO_DIFIERE (primeros 8 digitos): 0.1% difieren
REGIMEN_LGCP: 54.9% bajo Ley 9986

Features derivadas creadas:
  LOG_PRECIO_OFERTADO                  3,008,179 no-null (99.6%)
  LOG_PRECIO_ESTIMADO                  2,996,249 no-null (99.2%)
  LOG_RATIO_VS_ESTIMADO                2,991,968 no-null (99.0%)
  DIAS_PARA_CIERRE                     3,016,884 no-null (99.9%)
  DIAS_VENTANA_PROCESO                 3,016,884 no-null (99.9%)
  RATIO_CANTIDAD_VS_SOLICITADA         3,008,511 no-null (99.6%)
  PRODUCTO_DIFIERE                     3,021,336 no-null (100.0%)
  CV_PRECIO_ITEM                       2,629,398 no-null (87.0%)


  REGIMEN_LGCP                         3,021,336 no-null (100.0%)


In [4]:
# CORRECCION: TAMANO_PROVEEDOR incluye Microemprendedor
tamano_map = {'Microemprendedor': 1, 'Peque\u00f1a': 2, 'Mediana': 3, 'Grande': 4}
df['TAMANO_PROVEEDOR_ORD'] = df['TAMANO_PROVEEDOR'].map(tamano_map)
unmapped = df['TAMANO_PROVEEDOR'][df['TAMANO_PROVEEDOR_ORD'].isna()].unique()
print(f'TAMANO_PROVEEDOR mapeado: Microemprendedor=1, Pequena=2, Mediana=3, Grande=4')
if len(unmapped) > 0:
    print(f'Valores sin mapear (quedan NaN): {unmapped}')
    n_unmapped = df['TAMANO_PROVEEDOR_ORD'].isna().sum()
    print(f'Registros sin mapear: {n_unmapped:,} ({n_unmapped/len(df)*100:.2f}%)')
else:
    print('Todos los valores mapeados correctamente.')

print(f'\nTIPO_PROCEDIMIENTO: {df["TIPO_PROCEDIMIENTO"].nunique()} valores')
print(df['TIPO_PROCEDIMIENTO'].value_counts())
tp_dummies = pd.get_dummies(df['TIPO_PROCEDIMIENTO'], prefix='TP', dummy_na=False).astype(np.int8)
for c in tp_dummies.columns:
    df[c] = tp_dummies[c].values
del tp_dummies

print(f'\nMODALIDAD_PROCEDIMIENTO: {df["MODALIDAD_PROCEDIMIENTO"].nunique()} valores')
mp_dummies = pd.get_dummies(df['MODALIDAD_PROCEDIMIENTO'], prefix='MP', dummy_na=False).astype(np.int8)
for c in mp_dummies.columns:
    df[c] = mp_dummies[c].values
del mp_dummies

print(f'\nColumnas totales: {df.shape[1]}')

TAMANO_PROVEEDOR mapeado: Microemprendedor=1, Pequena=2, Mediana=3, Grande=4
Valores sin mapear (quedan NaN): ['No clasificado']
Registros sin mapear: 1,563 (0.05%)

TIPO_PROCEDIMIENTO: 12 valores


TIPO_PROCEDIMIENTO
Licitación Reducida                 1050805
Contratación Directa                1035200
Licitación Menor                     264423
Licitación Abreviada                 223709
Licitación Mayor                     184301
Licitación Pública Nacional           81852
Procedimientos Especiales             64311
Procedimiento por Excepción           51654
Contratación Especial                 33262
Procedimiento por Principio           27025
Licitación Pública Internacional        408
Subasta Inversa Electrónica              61
Name: count, dtype: int64



MODALIDAD_PROCEDIMIENTO: 20 valores



Columnas totales: 112


In [5]:
# Definicion de features: SIN variables de resultado
tp_cols = [c for c in df.columns if c.startswith('TP_')]
mp_cols = [c for c in df.columns if c.startswith('MP_')]

FEATURES_RELATIVAS = [
    'RATIO_OFERTADO_VS_ESTIMADO', 'RATIO_OFERTADO_VS_PROM_ITEM',
    'RATIO_OFERTADO_VS_MIN_ITEM', 'LOG_RATIO_VS_ESTIMADO',
    'RANK_PRECIO_ASC', 'N_OFERTAS_ITEM', 'N_OFERENTES_ITEM',
    'N_OFERENTES_ELEGIBLES_ITEM', 'SINGLE_BID_FLAG', 'CV_PRECIO_ITEM',
    'RATIO_CANTIDAD_VS_SOLICITADA', 'PRODUCTO_DIFIERE',
    'DIAS_PARA_CIERRE', 'DIAS_VENTANA_PROCESO',
    'OFERTA_IVA', 'OFERTA_DESCUENTO', 'REGIMEN_LGCP',
] + tp_cols + mp_cols + ['TAMANO_PROVEEDOR_ORD']

FEATURES_ABSOLUTAS = [
    'LOG_PRECIO_OFERTADO', 'LOG_PRECIO_ESTIMADO', 'CANTIDAD_OFERTADA',
    'N_OFERTAS_ITEM', 'N_OFERENTES_ITEM', 'N_OFERENTES_ELEGIBLES_ITEM',
    'SINGLE_BID_FLAG', 'RATIO_OFERTADO_VS_ESTIMADO', 'LOG_RATIO_VS_ESTIMADO',
    'CV_PRECIO_ITEM', 'RATIO_CANTIDAD_VS_SOLICITADA', 'PRODUCTO_DIFIERE',
    'DIAS_PARA_CIERRE', 'DIAS_VENTANA_PROCESO',
    'OFERTA_IVA', 'OFERTA_DESCUENTO', 'REGIMEN_LGCP',
] + tp_cols + mp_cols + ['TAMANO_PROVEEDOR_ORD']

print(f'Features RELATIVAS (Nivel 1): {len(FEATURES_RELATIVAS)}')
print(f'Features ABSOLUTAS (Nivel 2): {len(FEATURES_ABSOLUTAS)}')
print('\nVariables EXCLUIDAS respecto a exp_01:')
print('  - FUE_ADJUDICADO (variable de resultado, data leakage)')
print('  - CONTRATO_PRECIO_UNITARIO_CRC (no disponible ex ante)')
print('  - CONTRATO_CANTIDAD (no disponible ex ante)')
print('  - CONTRATO_TIPO_MONEDA (no disponible ex ante)')
print('\nVariables AGREGADAS respecto a exp_01:')
print('  + REGIMEN_LGCP (flag Ley 7494 vs Ley 9986)')

Features RELATIVAS (Nivel 1): 50
Features ABSOLUTAS (Nivel 2): 50

Variables EXCLUIDAS respecto a exp_01:
  - FUE_ADJUDICADO (variable de resultado, data leakage)
  - CONTRATO_PRECIO_UNITARIO_CRC (no disponible ex ante)
  - CONTRATO_CANTIDAD (no disponible ex ante)
  - CONTRATO_TIPO_MONEDA (no disponible ex ante)

Variables AGREGADAS respecto a exp_01:
  + REGIMEN_LGCP (flag Ley 7494 vs Ley 9986)


In [6]:
# Imputacion de nulos
all_features = list(set(FEATURES_RELATIVAS + FEATURES_ABSOLUTAS))

print('Nulos antes de imputacion:')
for f in sorted(all_features):
    n_null = df[f].isna().sum()
    if n_null > 0:
        print(f'  {f:<40s} {n_null:>10,} ({n_null/len(df)*100:.2f}%)')

binary_features = ['SINGLE_BID_FLAG', 'PRODUCTO_DIFIERE', 'REGIMEN_LGCP'] + tp_cols + mp_cols
for f in binary_features:
    if f in df.columns:
        df[f] = df[f].fillna(0)

numeric_features = [f for f in all_features if f not in binary_features]
for f in numeric_features:
    if f in df.columns and df[f].isna().any():
        df[f] = df[f].fillna(df[f].median())

if 'RANK_PRECIO_ASC' in df.columns:
    df['RANK_PRECIO_ASC'] = df['RANK_PRECIO_ASC'].astype(float)

remaining = df[all_features].isna().sum().sum()
print(f'\nNulos restantes: {remaining}')

Nulos antes de imputacion:
  CV_PRECIO_ITEM                              391,938 (12.97%)
  DIAS_PARA_CIERRE                              4,452 (0.15%)
  DIAS_VENTANA_PROCESO                          4,452 (0.15%)
  LOG_PRECIO_ESTIMADO                          25,087 (0.83%)
  LOG_PRECIO_OFERTADO                          13,157 (0.44%)
  LOG_RATIO_VS_ESTIMADO                        29,368 (0.97%)


  OFERTA_DESCUENTO                                897 (0.03%)
  OFERTA_IVA                                      933 (0.03%)
  RANK_PRECIO_ASC                                 886 (0.03%)
  RATIO_CANTIDAD_VS_SOLICITADA                 12,825 (0.42%)
  RATIO_OFERTADO_VS_ESTIMADO                   25,971 (0.86%)
  RATIO_OFERTADO_VS_MIN_ITEM                   18,972 (0.63%)
  RATIO_OFERTADO_VS_PROM_ITEM                  10,762 (0.36%)
  TAMANO_PROVEEDOR_ORD                          1,563 (0.05%)



Nulos restantes: 0


## 3. Modelo Nivel 1: por Segmento UNSPSC (semilla principal)

In [7]:
IF_PARAMS = dict(
    n_estimators=200, max_samples='auto',
    contamination='auto', random_state=SEMILLA_PRINCIPAL, n_jobs=-1,
)

segmentos = sorted(df['OFERTA_SEGMENTO'].unique())
df['SCORE_SEGMENTO'] = np.nan
seg_stats = []

print(f'Entrenando {len(segmentos)} modelos por segmento (seed={SEMILLA_PRINCIPAL})...')
print(f'{"Seg":>4s}  {"N":>10s}  {"Min":>10s}  {"Med":>10s}  {"Max":>10s}  {"Anom(auto)":>12s}')
print('-' * 65)
t0 = time.time()

for seg in segmentos:
    mask = df['OFERTA_SEGMENTO'] == seg
    X = df.loc[mask, FEATURES_RELATIVAS].values
    n = X.shape[0]
    if n < 10:
        print(f'{seg:>4s}  {n:>10,}  (omitido)')
        continue
    model = IsolationForest(**IF_PARAMS)
    model.fit(X)
    scores = model.decision_function(X)
    labels = model.predict(X)
    df.loc[mask, 'SCORE_SEGMENTO'] = scores
    n_anom = (labels == -1).sum()
    print(f'{seg:>4s}  {n:>10,}  {scores.min():>10.4f}  {np.median(scores):>10.4f}  {scores.max():>10.4f}  {n_anom:>7,} ({n_anom/n*100:.1f}%)')
    seg_stats.append({'segmento': seg, 'n': n, 'score_min': scores.min(),
                      'score_median': np.median(scores), 'score_max': scores.max(),
                      'n_anomalias': n_anom, 'pct_anomalias': n_anom/n*100})

seg_stats_df = pd.DataFrame(seg_stats)
print(f'\nNivel 1 completado en {time.time()-t0:.1f}s')
print(f'Modelos: {len(seg_stats)}  |  Ofertas con score: {df["SCORE_SEGMENTO"].notna().sum():,}')

Entrenando 57 modelos por segmento (seed=42)...
 Seg           N         Min         Med         Max    Anom(auto)
-----------------------------------------------------------------


  10      20,446     -0.1192      0.0935      0.1436    1,132 (5.5%)


  11      26,698     -0.1879      0.0977      0.1477    1,987 (7.4%)


  12      12,393     -0.0904      0.0906      0.1448      377 (3.0%)


  13         938     -0.1273      0.0941      0.1533       63 (6.7%)


...

  91       2,474     -0.1460      0.0770      0.1417      152 (6.1%)


  92      14,794     -0.1130      0.0814      0.1525      535 (3.6%)


  93       3,761     -0.1756      0.0822      0.1359      341 (9.1%)


  94         194     -0.1453      0.0974      0.1481       29 (14.9%)


  95       1,477     -0.1957      0.0927      0.1477      119 (8.1%)

Nivel 1 completado en 260.4s
Modelos: 57  |  Ofertas con score: 3,021,336


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].hist(df['SCORE_SEGMENTO'].dropna(), bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Score Segmento')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribucion de scores - Nivel 1 (Segmento)')
axes[0].axvline(x=0, color='red', linestyle='--', label='Umbral 0')
axes[0].legend()

top10 = seg_stats_df.nlargest(10, 'n')['segmento'].tolist()
data_box = [df.loc[df['OFERTA_SEGMENTO']==s, 'SCORE_SEGMENTO'].dropna().values for s in top10]
axes[1].boxplot(data_box, labels=top10, vert=True, patch_artist=True)
axes[1].set_xlabel('Segmento UNSPSC')
axes[1].set_ylabel('Score')
axes[1].set_title('Scores por segmento (top 10 por volumen)')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(FIG_DIR / 'scores_nivel1_segmento.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Modelo Nivel 2: por Clase UNSPSC (semilla principal)

In [9]:
clase_counts = df['OFERTA_CLASE'].value_counts()
clases_viables = clase_counts[clase_counts >= MIN_OBS_CLASE].index.tolist()

df['SCORE_CLASE'] = np.nan
IF_PARAMS_CLASE = {**IF_PARAMS, 'n_jobs': 1}

clase_stats = []
print(f'Entrenando {len(clases_viables)} modelos por clase (n >= {MIN_OBS_CLASE}, seed={SEMILLA_PRINCIPAL})...')
t0 = time.time()

for i, clase in enumerate(sorted(clases_viables)):
    mask = df['OFERTA_CLASE'] == clase
    X = df.loc[mask, FEATURES_ABSOLUTAS].values
    n = X.shape[0]
    model = IsolationForest(**IF_PARAMS_CLASE)
    model.fit(X)
    scores = model.decision_function(X)
    df.loc[mask, 'SCORE_CLASE'] = scores
    n_anom = (model.predict(X) == -1).sum()
    clase_stats.append({'clase': clase, 'n': n, 'score_min': scores.min(),
                        'score_median': np.median(scores), 'n_anomalias': n_anom})
    if (i + 1) % 200 == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (len(clases_viables) - i - 1) / rate
        print(f'  {i+1}/{len(clases_viables)} ({elapsed:.0f}s, ETA {eta:.0f}s)')

elapsed_total = time.time() - t0
clase_stats_df = pd.DataFrame(clase_stats)
n_con_clase = df['SCORE_CLASE'].notna().sum()
print(f'\nNivel 2 completado en {elapsed_total:.1f}s')
print(f'Modelos: {len(clase_stats)}')
print(f'Ofertas con SCORE_CLASE: {n_con_clase:,} ({n_con_clase/len(df)*100:.1f}%)')
print(f'Fallback a segmento: {len(df)-n_con_clase:,} ({(len(df)-n_con_clase)/len(df)*100:.1f}%)')

Entrenando 656 modelos por clase (n >= 500, seed=42)...


  200/656 (355s, ETA 810s)


  400/656 (697s, ETA 446s)


  600/656 (1017s, ETA 95s)



Nivel 2 completado en 1107.5s
Modelos: 656
Ofertas con SCORE_CLASE: 2,862,578 (94.7%)
Fallback a segmento: 158,758 (5.3%)


## 5. Combinacion de scores: tres reglas de agregacion

In [10]:
has_clase = df['SCORE_CLASE'].notna()

# Regla 1: min (conservadora, usada en exp_01)
df['SCORE_MIN'] = np.where(has_clase,
    np.minimum(df['SCORE_SEGMENTO'], df['SCORE_CLASE']),
    df['SCORE_SEGMENTO'])

# Regla 2: mean (moderada)
df['SCORE_MEAN'] = np.where(has_clase,
    (df['SCORE_SEGMENTO'] + df['SCORE_CLASE']) / 2,
    df['SCORE_SEGMENTO'])

# Regla 3: max (restrictiva, ambos niveles deben señalar)
df['SCORE_MAX'] = np.where(has_clase,
    np.maximum(df['SCORE_SEGMENTO'], df['SCORE_CLASE']),
    df['SCORE_SEGMENTO'])

# Z-score normalizado por segmento (para comparar scores entre segmentos)
df['ZSCORE_SEGMENTO'] = df.groupby('OFERTA_SEGMENTO')['SCORE_SEGMENTO'].transform(
    lambda x: (x - x.mean()) / x.std())
df['ZSCORE_CLASE'] = np.nan
for clase in clases_viables:
    mask = df['OFERTA_CLASE'] == clase
    vals = df.loc[mask, 'SCORE_CLASE']
    if vals.std() > 0:
        df.loc[mask, 'ZSCORE_CLASE'] = (vals - vals.mean()) / vals.std()
df['SCORE_ZNORM'] = np.where(df['ZSCORE_CLASE'].notna(),
    np.minimum(df['ZSCORE_SEGMENTO'], df['ZSCORE_CLASE']),
    df['ZSCORE_SEGMENTO'])

print('Scores combinados calculados:')
for col in ['SCORE_MIN', 'SCORE_MEAN', 'SCORE_MAX', 'SCORE_ZNORM']:
    q5 = df[col].quantile(0.05)
    print(f'  {col:<18s}  P5={q5:.4f}  min={df[col].min():.4f}  med={df[col].median():.4f}  max={df[col].max():.4f}')

Scores combinados calculados:
  SCORE_MIN           P5=-0.0217  min=-0.2828  med=0.0832  max=0.1594


  SCORE_MEAN          P5=-0.0012  min=-0.2353  med=0.0933  max=0.1601
  SCORE_MAX           P5=0.0138  min=-0.2036  med=0.1046  max=0.1709


  SCORE_ZNORM         P5=-2.2459  min=-8.9346  med=0.0220  max=1.6579


## 6. Analisis de umbrales multiples

In [11]:
PERCENTILES = [0.01, 0.02, 0.03, 0.05, 0.10]
SCORE_COLS = ['SCORE_MIN', 'SCORE_MEAN', 'SCORE_MAX', 'SCORE_ZNORM']

threshold_results = []
for score_col in SCORE_COLS:
    for pct in PERCENTILES:
        umbral = df[score_col].quantile(pct)
        flagged = df[score_col] <= umbral
        n_flag = flagged.sum()
        adj_rate = df.loc[flagged, 'FUE_ADJUDICADO'].mean() * 100 if n_flag > 0 else 0
        sb_rate = df.loc[flagged, 'SINGLE_BID_FLAG'].mean() * 100 if n_flag > 0 else 0
        threshold_results.append({
            'score': score_col, 'percentil': f'P{int(pct*100)}',
            'umbral': umbral, 'n_anomalias': n_flag,
            'pct_dataset': n_flag/len(df)*100,
            'pct_adjudicadas': adj_rate,
            'pct_single_bid': sb_rate,
        })

th_df = pd.DataFrame(threshold_results)
print('Anomalias detectadas por score y umbral:\n')
for sc in SCORE_COLS:
    print(f'--- {sc} ---')
    sub = th_df[th_df['score'] == sc]
    print(f'{"Pctil":>6s} {"Umbral":>10s} {"N anom":>10s} {"% dataset":>10s} {"% adj":>7s} {"% SB":>7s}')
    for _, r in sub.iterrows():
        print(f'{r["percentil"]:>6s} {r["umbral"]:>10.4f} {int(r["n_anomalias"]):>10,} {r["pct_dataset"]:>9.1f}% {r["pct_adjudicadas"]:>6.1f}% {r["pct_single_bid"]:>6.1f}%')
    print()

Anomalias detectadas por score y umbral:

--- SCORE_MIN ---
 Pctil     Umbral     N anom  % dataset   % adj    % SB
    P1    -0.0742     30,214       1.0%   27.1%   22.2%
    P2    -0.0528     60,427       2.0%   27.1%   21.7%
    P3    -0.0397     90,641       3.0%   27.3%   22.0%
    P5    -0.0217    151,067       5.0%   28.0%   22.2%
   P10     0.0050    302,134      10.0%   28.6%   21.5%

--- SCORE_MEAN ---
 Pctil     Umbral     N anom  % dataset   % adj    % SB
    P1    -0.0501     30,214       1.0%   24.4%   19.3%
    P2    -0.0301     60,427       2.0%   25.4%   20.3%
    P3    -0.0177     90,645       3.0%   26.3%   20.9%
    P5    -0.0012    151,068       5.0%   27.4%   21.2%
   P10     0.0234    302,134      10.0%   28.0%   20.8%

--- SCORE_MAX ---
 Pctil     Umbral     N anom  % dataset   % adj    % SB
    P1    -0.0346     30,214       1.0%   23.2%   18.0%
    P2    -0.0150     60,433       2.0%   24.8%   19.6%
    P3    -0.0028     90,641       3.0%   25.7%   20.1%
    P

In [12]:
# Overlap entre umbrales para SCORE_MIN
print('Overlap de anomalias entre umbrales (SCORE_MIN):\n')
flags = {}
for pct in PERCENTILES:
    umbral = df['SCORE_MIN'].quantile(pct)
    flags[f'P{int(pct*100)}'] = set(df[df['SCORE_MIN'] <= umbral].index)

print(f'{"":>6s}', end='')
for p in flags:
    print(f'{p:>10s}', end='')
print()
for p1, s1 in flags.items():
    print(f'{p1:>6s}', end='')
    for p2, s2 in flags.items():
        if len(s1) > 0 and len(s2) > 0:
            jaccard = len(s1 & s2) / len(s1 | s2)
            print(f'{jaccard:>10.3f}', end='')
        else:
            print(f'{0:>10.3f}', end='')
    print()

# Cuantas del P1 sobreviven en todos los umbrales
p1_set = flags['P1']
print(f'\nP1 tiene {len(p1_set):,} anomalias.')
for pname, pset in flags.items():
    overlap = len(p1_set & pset)
    print(f'  {overlap:,} ({overlap/len(p1_set)*100:.0f}%) de P1 estan en {pname}')

Overlap de anomalias entre umbrales (SCORE_MIN):



              P1        P2        P3        P5       P10
    P1     1.000     0.500     0.333     0.200     0.100
    P2     0.500     1.000     0.667     0.400

     0.200
    P3     0.333     0.667     1.000     0.600     0.300
    P5     0.200     0.400     0.600     1.000

     0.500
   P10     0.100     0.200     0.300     0.500     1.000

P1 tiene 30,214 anomalias.
  30,214 (100%) de P1 estan en P1
  30,214 (100%) de P1 estan en P2
  30,214 (100%) de P1 estan en P3
  30,214 (100%) de P1 estan en P5
  30,214 (100%) de P1 estan en P10


In [13]:
# Overlap entre reglas de agregacion (al P5)
print('Overlap entre reglas de agregacion al P5:\n')
rule_flags = {}
for sc in SCORE_COLS:
    umbral = df[sc].quantile(0.05)
    rule_flags[sc.replace('SCORE_','')] = set(df[df[sc] <= umbral].index)

print(f'{"":>10s}', end='')
for r in rule_flags:
    print(f'{r:>10s}', end='')
print()
for r1, s1 in rule_flags.items():
    print(f'{r1:>10s}', end='')
    for r2, s2 in rule_flags.items():
        jaccard = len(s1 & s2) / len(s1 | s2) if len(s1 | s2) > 0 else 0
        print(f'{jaccard:>10.3f}', end='')
    print()

# Anomalias consenso: detectadas por las 3 reglas crudas
consensus = rule_flags['MIN'] & rule_flags['MEAN'] & rule_flags['MAX']
print(f'\nAnomalias consenso (MIN ∩ MEAN ∩ MAX): {len(consensus):,} ({len(consensus)/len(df)*100:.2f}%)')

fig, ax = plt.subplots(figsize=(10, 6))
rules = list(rule_flags.keys())
sizes = [len(rule_flags[r]) for r in rules]
ax.bar(rules, sizes, color=['coral', 'steelblue', 'seagreen', 'orchid'])
ax.axhline(y=len(consensus), color='red', linestyle='--', label=f'Consenso: {len(consensus):,}')
ax.set_ylabel('N anomalias al P5')
ax.set_title('Anomalias detectadas por regla de agregacion (P5)')
ax.legend()
for i, (r, s) in enumerate(zip(rules, sizes)):
    ax.text(i, s + 1000, f'{s:,}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig(FIG_DIR / 'comparacion_reglas_agregacion.png', dpi=150, bbox_inches='tight')
plt.show()

Overlap entre reglas de agregacion al P5:



                 MIN      MEAN       MAX     ZNORM
       MIN     1.000     0.673     0.479     0.695
      MEAN

     0.673     1.000     0.723     0.667
       MAX     0.479     0.723     1.000     0.518
     ZNORM

     0.695     0.667     0.518     1.000

Anomalias consenso (MIN ∩ MEAN ∩ MAX): 97,889 (3.24%)


## 7. Resultados con SCORE_MIN al P5 (comparable con exp_01)

In [14]:
# Flag principal: SCORE_MIN al P5
umbral_p5 = df['SCORE_MIN'].quantile(0.05)
df['ANOMALY_FLAG'] = (df['SCORE_MIN'] <= umbral_p5).astype(int)
n_anom = df['ANOMALY_FLAG'].sum()
anom = df[df['ANOMALY_FLAG']==1]
normal = df[df['ANOMALY_FLAG']==0]

print(f'Umbral (P5 de SCORE_MIN): {umbral_p5:.4f}')
print(f'Anomalias detectadas: {n_anom:,} ({n_anom/len(df)*100:.1f}%)')
print(f'\nDistribucion de SCORE_MIN:')
print(df['SCORE_MIN'].describe())

Umbral (P5 de SCORE_MIN): -0.0217
Anomalias detectadas: 151,067 (5.0%)

Distribucion de SCORE_MIN:


count    3.021336e+06
mean     7.336773e-02
std      4.937568e-02
min     -2.827638e-01
25%      4.493396e-02
50%      8.317287e-02
75%      1.117777e-01
max      1.594115e-01
Name: SCORE_MIN, dtype: float64


In [15]:
# Perfil comparativo: medias, medianas, P25, P75
compare_cols = [
    'OFERTA_PRECIO_UNITARIO_CRC', 'RATIO_OFERTADO_VS_ESTIMADO',
    'RATIO_OFERTADO_VS_PROM_ITEM', 'N_OFERENTES_ITEM',
    'DIAS_PARA_CIERRE', 'CV_PRECIO_ITEM',
    'SINGLE_BID_FLAG', 'PRODUCTO_DIFIERE',
]

print(f'Anomalias: {len(anom):,}  |  Normales: {len(normal):,}')
print(f'\n{"Feature":<35s} {"Media A":>12s} {"Media N":>12s} {"R(med)":>8s} {"Med A":>12s} {"Med N":>12s} {"R(med)":>8s}')
print('-' * 100)
profile_rows = []
for col in compare_cols:
    m_a = anom[col].mean(); m_n = normal[col].mean()
    md_a = anom[col].median(); md_n = normal[col].median()
    r_mean = m_a / m_n if m_n != 0 else np.nan
    r_med = md_a / md_n if md_n != 0 else np.nan
    p25_a = anom[col].quantile(0.25); p75_a = anom[col].quantile(0.75)
    p25_n = normal[col].quantile(0.25); p75_n = normal[col].quantile(0.75)
    print(f'{col:<35s} {m_a:>12.2f} {m_n:>12.2f} {r_mean:>7.2f}x {md_a:>12.2f} {md_n:>12.2f} {r_med:>7.2f}x')
    profile_rows.append({'feature': col, 'media_anom': m_a, 'media_norm': m_n, 'ratio_media': r_mean,
                         'mediana_anom': md_a, 'mediana_norm': md_n, 'ratio_mediana': r_med,
                         'p25_anom': p25_a, 'p75_anom': p75_a, 'p25_norm': p25_n, 'p75_norm': p75_n})
profile_df = pd.DataFrame(profile_rows)

# Analisis post-hoc con FUE_ADJUDICADO (NO fue feature, pero es variable de resultado)
adj_anom = anom['FUE_ADJUDICADO'].mean() * 100
adj_norm = normal['FUE_ADJUDICADO'].mean() * 100
n_adj_total = (df['FUE_ADJUDICADO']==1).sum()
n_adj_anom = (anom['FUE_ADJUDICADO']==1).sum()
print(f'\nAnalisis post-hoc (FUE_ADJUDICADO NO fue feature):')
print(f'  % adjudicadas en anomalias: {adj_anom:.1f}%  |  en normales: {adj_norm:.1f}%  |  ratio: {adj_anom/adj_norm:.2f}x')
print(f'  Anomalias adjudicadas: {n_adj_anom:,} / {n_adj_total:,} adjudicaciones totales ({n_adj_anom/n_adj_total*100:.1f}%)')

Anomalias: 151,067  |  Normales: 2,870,269

Feature                                  Media A      Media N   R(med)        Med A        Med N   R(med)
----------------------------------------------------------------------------------------------------


OFERTA_PRECIO_UNITARIO_CRC          303718461.93   5178443.68   58.65x     18000.00     15695.00    1.15x


RATIO_OFERTADO_VS_ESTIMADO           13400992.02     16901.40  792.89x         1.00         0.91    1.10x


RATIO_OFERTADO_VS_PROM_ITEM                 1.21         0.99    1.22x         1.00         1.00    1.00x


N_OFERENTES_ITEM                            6.00         4.46    1.34x         4.00         4.00    1.00x


DIAS_PARA_CIERRE                            2.79         0.96    2.90x         0.64         0.46    1.39x


CV_PRECIO_ITEM                              0.67         0.39    1.71x         0.32         0.32    1.00x
SINGLE_BID_FLAG                             0.22         0.12    1.82x         0.00         0.00     nanx


PRODUCTO_DIFIERE                            0.00         0.00    0.58x         0.00         0.00     nanx

Analisis post-hoc (FUE_ADJUDICADO NO fue feature):
  % adjudicadas en anomalias: 28.0%  |  en normales: 30.3%  |  ratio: 0.93x
  Anomalias adjudicadas: 42,364 / 911,493 adjudicaciones totales (4.6%)


In [16]:
# Anomalias por tipo de procedimiento
anom_by_tp = df.groupby('TIPO_PROCEDIMIENTO').agg(
    total=('ANOMALY_FLAG','count'), anomalias=('ANOMALY_FLAG','sum')
).reset_index()
anom_by_tp['pct'] = anom_by_tp['anomalias'] / anom_by_tp['total'] * 100
anom_by_tp = anom_by_tp.sort_values('pct', ascending=False)
print('Anomalias por tipo de procedimiento:')
print(anom_by_tp.to_string(index=False))

# Anomalias por tamano de proveedor
anom_by_tam = df.groupby('TAMANO_PROVEEDOR').agg(
    total=('ANOMALY_FLAG','count'), anomalias=('ANOMALY_FLAG','sum')
).reset_index()
anom_by_tam['pct'] = anom_by_tam['anomalias'] / anom_by_tam['total'] * 100
print('\nAnomalias por tamano de proveedor:')
print(anom_by_tam.to_string(index=False))

Anomalias por tipo de procedimiento:
              TIPO_PROCEDIMIENTO   total  anomalias       pct
           Contratación Especial   33262      11357 34.144068
     Licitación Pública Nacional   81852      17231 21.051410
                Licitación Mayor  184301      24532 13.310834
            Licitación Abreviada  223709      22520 10.066649
Licitación Pública Internacional     408         39  9.558824
                Licitación Menor  264423      24488  9.260919
     Procedimiento por Principio   27025       2457  9.091582
       Procedimientos Especiales   64311       4608  7.165182
     Procedimiento por Excepción   51654       2898  5.610408
            Contratación Directa 1035200      22769  2.199478
             Licitación Reducida 1050805      18040  1.716779
     Subasta Inversa Electrónica      61          0  0.000000



Anomalias por tamano de proveedor:
TAMANO_PROVEEDOR  total  anomalias      pct
          Grande 847659      46484 5.483809
         Mediana 770055      37631 4.886794
Microemprendedor 470712      24231 5.147734
  No clasificado   1563         87 5.566219
         Pequeña 931347      42634 4.577671


In [17]:
# Reporte estratificado por segmento
anom_by_seg = df.groupby('OFERTA_SEGMENTO').agg(
    total=('ANOMALY_FLAG','count'), anomalias=('ANOMALY_FLAG','sum'),
    pct_adj=('FUE_ADJUDICADO','mean'), pct_sb=('SINGLE_BID_FLAG','mean'),
    score_min=('SCORE_MIN','min'), score_med=('SCORE_MIN','median'),
).reset_index()
anom_by_seg['pct_anom'] = anom_by_seg['anomalias'] / anom_by_seg['total'] * 100
anom_by_seg['pct_del_total_anom'] = anom_by_seg['anomalias'] / n_anom * 100
anom_by_seg = anom_by_seg.sort_values('anomalias', ascending=False)

print('Reporte estratificado por segmento (top 20):\n')
print(f'{"Seg":>4s} {"Total":>9s} {"Anom":>7s} {"Tasa%":>7s} {"% del total":>11s} {"SB%":>6s} {"Score min":>10s}')
print('-' * 60)
for _, r in anom_by_seg.head(20).iterrows():
    print(f'{r["OFERTA_SEGMENTO"]:>4s} {int(r["total"]):>9,} {int(r["anomalias"]):>7,} {r["pct_anom"]:>6.1f}% {r["pct_del_total_anom"]:>10.1f}% {r["pct_sb"]*100:>5.0f}% {r["score_min"]:>10.4f}')

# Verificacion de dominancia
top3_share = anom_by_seg.head(3)['pct_del_total_anom'].sum()
top3_vol = anom_by_seg.head(3)['total'].sum() / len(df) * 100
print(f'\nTop 3 segmentos por anomalias concentran {top3_share:.1f}% de las anomalias (representan {top3_vol:.1f}% del volumen)')
if top3_share > top3_vol * 1.5:
    print('ALERTA: posible dominancia de segmentos en el perfil global.')

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
top15 = anom_by_seg.head(15)
axes[0].barh(top15['OFERTA_SEGMENTO'], top15['anomalias'], color='coral')
axes[0].set_xlabel('Cantidad de anomalias')
axes[0].set_title('Top 15 segmentos por cantidad de anomalias')
axes[0].invert_yaxis()

top15p = anom_by_seg.sort_values('pct_anom', ascending=False).head(15)
axes[1].barh(top15p['OFERTA_SEGMENTO'], top15p['pct_anom'], color='salmon')
axes[1].set_xlabel('% anomalias')
axes[1].set_title('Top 15 segmentos por tasa de anomalias')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(FIG_DIR / 'anomalias_por_segmento.png', dpi=150, bbox_inches='tight')
plt.show()

Reporte estratificado por segmento (top 20):

 Seg     Total    Anom   Tasa% % del total    SB%  Score min
------------------------------------------------------------
  42   190,003  10,892    5.7%        7.2%    19%    -0.2717
  30   174,094   9,493    5.5%        6.3%     5%    -0.2303
  31   174,396   8,159    4.7%        5.4%     7%    -0.1744
  44   170,968   7,862    4.6%        5.2%     6%    -0.1929
  27   123,213   7,832    6.4%        5.2%     6%    -0.1924
  40   179,359   7,710    4.3%        5.1%     9%    -0.2024
  41   123,457   6,447    5.2%        4.3%    17%    -0.1893
  72   202,651   6,066    3.0%        4.0%    11%    -0.1926
  47   139,288   5,842    4.2%        3.9%     3%    -0.2036
  78    91,872   5,579    6.1%        3.7%    42%    -0.2299
  55    56,362   5,262    9.3%        3.5%    15%    -0.2258
  46   109,049   4,715    4.3%        3.1%     6%    -0.1946
  43   100,188   4,669    4.7%        3.1%    12%    -0.1705
  83    23,337   3,978   17.0%        2

In [18]:
# Histogramas comparativos
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col, xlim in zip(axes.flat, [
    'RATIO_OFERTADO_VS_ESTIMADO', 'DIAS_PARA_CIERRE',
    'CV_PRECIO_ITEM', 'N_OFERENTES_ITEM',
    'LOG_PRECIO_OFERTADO', 'PRODUCTO_DIFIERE'
], [
    (0, 5), (-5, 30), (0, 3), (0, 20), (0, 12), (-0.5, 1.5)
]):
    bins = 80
    a_data = anom[col].clip(xlim[0], xlim[1]).dropna()
    n_data = normal[col].clip(xlim[0], xlim[1]).dropna()
    ax.hist(n_data, bins=bins, alpha=0.5, density=True, label='Normal', color='steelblue')
    ax.hist(a_data, bins=bins, alpha=0.5, density=True, label='Anomalia', color='coral')
    ax.set_xlabel(col)
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=8)
    ax.set_title(col)

plt.suptitle('Distribucion de features: anomalias vs normales (exp_02)', fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / 'features_anom_vs_normal.png', dpi=150, bbox_inches='tight')
plt.show()

In [19]:
# Top 30 anomalias
top30 = df.nsmallest(30, 'SCORE_MIN')[
    ['NRO_SICOP','NRO_OFERTA','NRO_LINEA',
     'OFERTA_SEGMENTO','OFERTA_CLASE',
     'NOMBRE_INSTITUCION','NOMBRE_PROVEEDOR','TAMANO_PROVEEDOR',
     'OFERTA_PRECIO_UNITARIO_CRC','CARTEL_PRECIO_UNITARIO_CRC',
     'RATIO_OFERTADO_VS_ESTIMADO','N_OFERENTES_ITEM',
     'SINGLE_BID_FLAG','FUE_ADJUDICADO','PRODUCTO_DIFIERE',
     'SCORE_SEGMENTO','SCORE_CLASE','SCORE_MIN']
].reset_index(drop=True)

print('Top 30 anomalias (SCORE_MIN):\n')
with pd.option_context('display.max_columns', None, 'display.width', 200,
                       'display.max_colwidth', 30, 'display.float_format', '{:.4f}'.format):
    print(top30.to_string())

Top 30 anomalias (SCORE_MIN):

      NRO_SICOP                         NRO_OFERTA  NRO_LINEA OFERTA_SEGMENTO OFERTA_CLASE                                                NOMBRE_INSTITUCION                                                 NOMBRE_PROVEEDOR  TAMANO_PROVEEDOR  OFERTA_PRECIO_UNITARIO_CRC  CARTEL_PRECIO_UNITARIO_CRC  RATIO_OFERTADO_VS_ESTIMADO  N_OFERENTES_ITEM  SINGLE_BID_FLAG  FUE_ADJUDICADO  PRODUCTO_DIFIERE  SCORE_SEGMENTO  SCORE_CLASE  SCORE_MIN
0   20240522855  D20240517175831310717159903118640          4              11       111618                                      MINISTERIO DE JUSTICIA Y PAZ                                       MICHAEL VILLAVICENCIO PEÑA           Pequeña                1487000.0000                   1803.0000                    824.7366                 8                0               0                 0         -0.1879      -0.2828    -0.2828
1   20211102488  D20211125085053484616378518536330          3              42       421429             

## 8. Caracterizacion de anomalias

In [20]:
# 8.1 Concentracion por proveedor
prov_anom = anom.groupby(['CEDULA_PROVEEDOR', 'NOMBRE_PROVEEDOR']).agg(
    anomalias=('ANOMALY_FLAG', 'sum'),
    score_min=('SCORE_MIN', 'min'),
    score_median=('SCORE_MIN', 'median'),
    segmentos_distintos=('OFERTA_SEGMENTO', 'nunique'),
    pct_adjudicado=('FUE_ADJUDICADO', 'mean'),
).reset_index()
prov_total = df.groupby('CEDULA_PROVEEDOR')['ANOMALY_FLAG'].agg(['count', 'sum']).reset_index()
prov_total.columns = ['CEDULA_PROVEEDOR', 'total_ofertas', 'total_anomalias']
prov_anom = prov_anom.merge(prov_total, on='CEDULA_PROVEEDOR')
prov_anom['tasa_anomalia'] = prov_anom['anomalias'] / prov_anom['total_ofertas'] * 100

top20_prov = prov_anom.nlargest(20, 'anomalias')
print('Top 20 proveedores por cantidad de anomalias:\n')
print(f'{"Proveedor":<45s} {"Anom":>6s} {"Total":>7s} {"Tasa%":>6s} {"Adj%":>5s} {"Segs":>4s}')
print('-' * 80)
for _, r in top20_prov.iterrows():
    nm = str(r['NOMBRE_PROVEEDOR'])[:44]
    print(f'{nm:<45s} {int(r["anomalias"]):>6,} {int(r["total_ofertas"]):>7,} {r["tasa_anomalia"]:>5.1f}% {r["pct_adjudicado"]*100:>4.0f}% {int(r["segmentos_distintos"]):>4}')

Top 20 proveedores por cantidad de anomalias:

Proveedor                                       Anom   Total  Tasa%  Adj% Segs
--------------------------------------------------------------------------------
CORPORACION QUIMISOL SOCIEDAD ANONIMA          8,502 151,419   5.6%    2%   40
INVERSIONES LA RUECA SOCIEDAD ANONIMA          2,595  98,717   2.6%   14%   45
REPRESENTACIONES SUMI COMP EQUIPOS SOCIEDAD    2,082  28,542   7.3%   17%   36
CORPORACION COMERCIAL EL LAGAR SOCIEDAD ANON   2,076  47,783   4.3%   27%   32
3-101-748418 SOCIEDAD ANONIMA                  1,808  44,396   4.1%   15%   32
JIMENEZ Y TANZI SOCIEDAD ANONIMA               1,718  14,607  11.8%   49%   26
G Y R GRUPO ASESOR, SOCIEDAD ANONIMA           1,361  34,640   3.9%    2%   44
ALMACENES EL COLONO SOCIEDAD ANONIMA           1,320  41,835   3.2%   33%   33
J ROBERTO VARGAS E HIJOS SOCIEDAD ANONIMA      1,205   7,562  15.9%   80%   20
MATERIALES Y FERRETERIA LA SUIZA SOCIEDAD AN   1,163  21,655   5.4%   20%   20
FES

In [21]:
# 8.2 Concentracion por institucion
inst_anom = anom.groupby(['CEDULA_INSTITUCION', 'NOMBRE_INSTITUCION']).agg(
    anomalias=('ANOMALY_FLAG', 'sum'),
    segmentos_distintos=('OFERTA_SEGMENTO', 'nunique'),
    proveedores_distintos=('CEDULA_PROVEEDOR', 'nunique'),
    pct_single_bid=('SINGLE_BID_FLAG', 'mean'),
).reset_index()
inst_total = df.groupby('CEDULA_INSTITUCION')['ANOMALY_FLAG'].agg(['count', 'sum']).reset_index()
inst_total.columns = ['CEDULA_INSTITUCION', 'total_ofertas', 'total_anomalias']
inst_anom = inst_anom.merge(inst_total, on='CEDULA_INSTITUCION')
inst_anom['tasa_anomalia'] = inst_anom['anomalias'] / inst_anom['total_ofertas'] * 100

top20_inst = inst_anom.nlargest(20, 'anomalias')
print('Top 20 instituciones por cantidad de anomalias:\n')
print(f'{"Institucion":<50s} {"Anom":>6s} {"Total":>7s} {"Tasa%":>6s} {"SB%":>5s} {"Provs":>5s}')
print('-' * 85)
for _, r in top20_inst.iterrows():
    nm = str(r['NOMBRE_INSTITUCION'])[:49]
    print(f'{nm:<50s} {int(r["anomalias"]):>6,} {int(r["total_ofertas"]):>7,} {r["tasa_anomalia"]:>5.1f}% {r["pct_single_bid"]*100:>4.0f}% {int(r["proveedores_distintos"]):>5}')

Top 20 instituciones por cantidad de anomalias:

Institucion                                          Anom   Total  Tasa%   SB% Provs
-------------------------------------------------------------------------------------
CAJA COSTARRICENSE DE SEGURO SOCIAL                23,441 472,566   5.0%   25%  1537
INSTITUTO NACIONAL DE SEGUROS                      10,005  68,752  14.6%   30%   643
INSTITUTO COSTARRICENSE DE ACUEDUCTOS Y ALCANTARI   6,755  64,952  10.4%   25%   641
INSTITUTO COSTARRICENSE DE ELECTRICIDAD             6,567 143,066   4.6%   28%   860
COMISIÓN NACIONAL DE PREVENCIÓN DE RIESGOS Y ATEN   5,316  30,879  17.2%   13%   595
INSTITUTO NACIONAL DE APRENDIZAJE                   4,366  73,027   6.0%   16%   765
UNIVERSIDAD DE COSTA RICA                           4,340 107,477   4.0%   12%   648
UNIVERSIDAD TÉCNICA NACIONAL                        3,377  30,244  11.2%   20%   288
MUNICIPALIDAD DEL CANTÓN CENTRAL DE SAN JOSÉ        3,040  71,446   4.3%   24%   384
MUNICIPALIDAD D

In [22]:
# 8.3 Patrones temporales
by_year = df.groupby(df['FECHA_PRESENTA_OFERTA'].dt.year).agg(
    total=('ANOMALY_FLAG', 'count'), anomalias=('ANOMALY_FLAG', 'sum')
).reset_index()
by_year.columns = ['anio', 'total', 'anomalias']
by_year['pct'] = by_year['anomalias'] / by_year['total'] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(by_year['anio'], by_year['anomalias'], color='coral', alpha=0.8)
axes[0].set_xlabel('Anio')
axes[0].set_ylabel('Cantidad de anomalias')
axes[0].set_title('Anomalias por anio')

axes[1].bar(by_year['anio'], by_year['pct'], color='salmon', alpha=0.8)
axes[1].set_xlabel('Anio')
axes[1].set_ylabel('% anomalias')
axes[1].set_title('Tasa de anomalias por anio')
axes[1].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='5%')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'anomalias_temporales.png', dpi=150, bbox_inches='tight')
plt.show()
print('Tasa de anomalias por anio:')
print(by_year.to_string(index=False))

Tasa de anomalias por anio:
 anio  total  anomalias      pct
 2020 418007      27206 6.508503
 2021 464944      22075 4.747884
 2022 504582      24507 4.856891
 2023 514795      18471 3.588030
 2024 565776      25817 4.563113
 2025 553232      32991 5.963321


## 9. Analisis de estabilidad (5 semillas)

In [23]:
def train_full_pipeline(dataframe, features_rel, features_abs, clases_list, seed, min_obs=500):
    """Entrena Nivel1 + Nivel2, retorna SCORE_MIN por oferta."""
    params_seg = dict(n_estimators=200, max_samples='auto',
                      contamination='auto', random_state=seed, n_jobs=-1)
    params_cls = {**params_seg, 'n_jobs': 1}

    scores_seg = np.full(len(dataframe), np.nan)
    scores_cls = np.full(len(dataframe), np.nan)

    for seg in dataframe['OFERTA_SEGMENTO'].unique():
        mask = (dataframe['OFERTA_SEGMENTO'] == seg).values
        X = dataframe.loc[mask, features_rel].values
        if X.shape[0] < 10:
            continue
        m = IsolationForest(**params_seg)
        m.fit(X)
        scores_seg[mask] = m.decision_function(X)

    for clase in clases_list:
        mask = (dataframe['OFERTA_CLASE'] == clase).values
        X = dataframe.loc[mask, features_abs].values
        if X.shape[0] < min_obs:
            continue
        m = IsolationForest(**params_cls)
        m.fit(X)
        scores_cls[mask] = m.decision_function(X)

    has_cls = ~np.isnan(scores_cls)
    score_combined = np.where(has_cls, np.minimum(scores_seg, scores_cls), scores_seg)
    return score_combined

print(f'Ejecutando {len(SEMILLAS_ESTABILIDAD)} corridas de estabilidad...')
print(f'Semillas: {SEMILLAS_ESTABILIDAD}')
print(f'Cada corrida entrena {len(segmentos)} modelos de segmento + {len(clases_viables)} de clase.\n')

stability_scores = {}
stability_flags = {}

for seed in SEMILLAS_ESTABILIDAD:
    t0 = time.time()
    print(f'Semilla {seed}...', end=' ', flush=True)
    sc = train_full_pipeline(df, FEATURES_RELATIVAS, FEATURES_ABSOLUTAS,
                             sorted(clases_viables), seed, MIN_OBS_CLASE)
    umbral = np.nanpercentile(sc, 5)
    flag = (sc <= umbral).astype(int)
    stability_scores[seed] = sc
    stability_flags[seed] = flag
    n_flag = flag.sum()
    print(f'{time.time()-t0:.0f}s  |  umbral={umbral:.4f}  |  anomalias={n_flag:,} ({n_flag/len(df)*100:.1f}%)')

print('\nTodas las corridas completadas.')

Ejecutando 5 corridas de estabilidad...
Semillas: [42, 123, 456, 789, 2026]
Cada corrida entrena 57 modelos de segmento + 656 de clase.

Semilla 42... 

959s  |  umbral=-0.0217  |  anomalias=151,067 (5.0%)
Semilla 123... 

858s  |  umbral=-0.0240  |  anomalias=151,067 (5.0%)
Semilla 456... 

938s  |  umbral=-0.0212  |  anomalias=151,067 (5.0%)
Semilla 789... 

951s  |  umbral=-0.0220  |  anomalias=151,067 (5.0%)
Semilla 2026... 

937s  |  umbral=-0.0207  |  anomalias=151,067 (5.0%)

Todas las corridas completadas.


In [24]:
# Matriz de concordancia (Jaccard index)
seeds = list(stability_flags.keys())
print('Jaccard index entre pares de semillas (al P5):\n')
print(f'{"":>8s}', end='')
for s in seeds:
    print(f'{s:>8d}', end='')
print()

for s1 in seeds:
    print(f'{s1:>8d}', end='')
    f1 = set(np.where(stability_flags[s1] == 1)[0])
    for s2 in seeds:
        f2 = set(np.where(stability_flags[s2] == 1)[0])
        jacc = len(f1 & f2) / len(f1 | f2) if len(f1 | f2) > 0 else 0
        print(f'{jacc:>8.3f}', end='')
    print()

# Robustez por oferta
flag_matrix = np.column_stack([stability_flags[s] for s in seeds])
df['N_VECES_FLAGGED'] = flag_matrix.sum(axis=1)

print(f'\nDistribucion de robustez (de {len(seeds)} semillas):')
for k in range(len(seeds)+1):
    n = (df['N_VECES_FLAGGED'] == k).sum()
    pct = n / len(df) * 100
    label = ''
    if k == 0: label = '(nunca anomala)'
    elif k == len(seeds): label = '(ROBUSTA: siempre anomala)'
    elif k == 1: label = '(fragil)'
    print(f'  {k}/{len(seeds)} veces: {n:>10,} ({pct:>5.2f}%) {label}')

n_robust = (df['N_VECES_FLAGGED'] == len(seeds)).sum()
n_fragile = (df['N_VECES_FLAGGED'] == 1).sum()
print(f'\nAnomalias robustas ({len(seeds)}/{len(seeds)}): {n_robust:,} ({n_robust/len(df)*100:.2f}%)')
print(f'Anomalias fragiles (1/{len(seeds)}): {n_fragile:,} ({n_fragile/len(df)*100:.2f}%)')

Jaccard index entre pares de semillas (al P5):

              42     123     456     789    2026
      42   1.000

   0.730

   0.752   0.725

   0.752
     123

   0.730

...

   0.747

   0.759   0.736

   1.000



Distribucion de robustez (de 5 semillas):
  0/5 veces:  2,820,383 (93.35%) (nunca anomala)
  1/5 veces:     32,255 ( 1.07%) (fragil)
  2/5 veces:     20,677 ( 0.68%) 
  3/5 veces:     18,589 ( 0.62%) 
  4/5 veces:     21,201 ( 0.70%) 


  5/5 veces:    108,231 ( 3.58%) (ROBUSTA: siempre anomala)

Anomalias robustas (5/5): 108,231 (3.58%)
Anomalias fragiles (1/5): 32,255 (1.07%)


In [25]:
# Visualizacion de estabilidad
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

counts = [((df['N_VECES_FLAGGED'] == k).sum()) for k in range(len(seeds)+1)]
colors = ['steelblue'] + ['gold']*(len(seeds)-2) + ['coral', 'darkred']
axes[0].bar(range(len(seeds)+1), counts, color=colors[:len(counts)])
axes[0].set_xlabel(f'Veces flaggeada (de {len(seeds)} semillas)')
axes[0].set_ylabel('Ofertas')
axes[0].set_title('Robustez de anomalias')
axes[0].set_xticks(range(len(seeds)+1))

robust_mask = df['N_VECES_FLAGGED'] == len(seeds)
fragile_mask = df['N_VECES_FLAGGED'] == 1
robust_scores = df.loc[robust_mask, 'SCORE_MIN'].dropna()
fragile_scores = df.loc[fragile_mask, 'SCORE_MIN'].dropna()
if len(robust_scores) > 0 and len(fragile_scores) > 0:
    axes[1].hist(robust_scores, bins=60, alpha=0.6, density=True, label=f'Robustas ({len(robust_scores):,})', color='darkred')
    axes[1].hist(fragile_scores, bins=60, alpha=0.6, density=True, label=f'Fragiles ({len(fragile_scores):,})', color='gold')
    axes[1].set_xlabel('SCORE_MIN')
    axes[1].set_ylabel('Densidad')
    axes[1].set_title('Scores de anomalias robustas vs fragiles')
    axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'estabilidad_semillas.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Exportar resultados

In [26]:
export_cols = [
    'NRO_SICOP','NRO_PROCEDIMIENTO','NRO_OFERTA','NRO_LINEA',
    'CEDULA_INSTITUCION','NOMBRE_INSTITUCION',
    'TIPO_PROCEDIMIENTO','MODALIDAD_PROCEDIMIENTO',
    'CEDULA_PROVEEDOR','NOMBRE_PROVEEDOR','TAMANO_PROVEEDOR',
    'OFERTA_SEGMENTO','OFERTA_FAMILIA','OFERTA_CLASE','OFERTA_MERCADERIA',
    'OFERTA_CODIGO_PRODUCTO','CARTEL_CODIGO_PRODUCTO',
    'OFERTA_TIPO_MONEDA','OFERTA_PRECIO_UNITARIO_CRC',
    'CARTEL_PRECIO_UNITARIO_CRC','CONTRATO_PRECIO_UNITARIO_CRC',
    'CANTIDAD_OFERTADA','CARTEL_CANTIDAD_SOLICITADA',
    'N_OFERTAS_ITEM','N_OFERENTES_ITEM','SINGLE_BID_FLAG','FUE_ADJUDICADO',
    'RATIO_OFERTADO_VS_ESTIMADO','RATIO_OFERTADO_VS_PROM_ITEM',
    'PRODUCTO_DIFIERE','REGIMEN_LGCP',
    'SCORE_SEGMENTO','SCORE_CLASE','SCORE_MIN','SCORE_MEAN','SCORE_MAX','SCORE_ZNORM',
    'ANOMALY_FLAG','N_VECES_FLAGGED',
]

out = df[export_cols].copy()
out.to_parquet(RES_DIR / 'anomalias_exp02.parquet', index=False)
size_mb = (RES_DIR / 'anomalias_exp02.parquet').stat().st_size / (1024*1024)
print(f'Guardado: resultados/anomalias_exp02.parquet ({size_mb:.1f} MB)')

anom_out = out[out['ANOMALY_FLAG']==1].sort_values('SCORE_MIN')
anom_out.to_csv(RES_DIR / 'anomalias_exp02_flagged.csv', index=False)
print(f'Anomalias: resultados/anomalias_exp02_flagged.csv ({len(anom_out):,} filas)')

robust_out = out[out['N_VECES_FLAGGED'] == len(SEMILLAS_ESTABILIDAD)].sort_values('SCORE_MIN')
robust_out.to_csv(RES_DIR / 'anomalias_exp02_robustas.csv', index=False)
print(f'Robustas: resultados/anomalias_exp02_robustas.csv ({len(robust_out):,} filas)')

Guardado: resultados/anomalias_exp02.parquet (320.6 MB)


Anomalias: resultados/anomalias_exp02_flagged.csv (151,067 filas)


Robustas: resultados/anomalias_exp02_robustas.csv (108,231 filas)


In [27]:
print('=' * 70)
print('RESUMEN EJECUTIVO - Experimento 02')
print('=' * 70)
print(f'Dataset:              {len(df):,} ofertas')
print(f'Segmentos UNSPSC:     {df["OFERTA_SEGMENTO"].nunique()}')
print(f'Clases UNSPSC:        {df["OFERTA_CLASE"].nunique()}')
print(f'Modelos Nivel 1:      {len(seg_stats)} (por segmento)')
print(f'Modelos Nivel 2:      {len(clase_stats)} (por clase, n >= {MIN_OBS_CLASE})')
print(f'Cobertura Nivel 2:    {n_con_clase:,} ({n_con_clase/len(df)*100:.1f}%)')
print(f'\nUmbral (P5 SCORE_MIN): {umbral_p5:.4f}')
print(f'Anomalias P5:          {n_anom:,} ({n_anom/len(df)*100:.1f}%)')
print(f'Anomalias robustas:    {n_robust:,} ({n_robust/len(df)*100:.2f}%)')
print(f'Anomalias fragiles:    {n_fragile:,} ({n_fragile/len(df)*100:.2f}%)')
adj_rate_posthoc = anom['FUE_ADJUDICADO'].mean() * 100
print(f'% anomalias adjudicadas (post-hoc): {adj_rate_posthoc:.1f}%')
print(f'\nCambios respecto a exp_01:')
print(f'  - FUE_ADJUDICADO excluida de features (era feature, ahora solo post-hoc)')
print(f'  - PRODUCTO_DIFIERE corregido (8 digitos): {pct_difiere:.1f}% difieren')
print(f'  - TAMANO_PROVEEDOR: Microemprendedor=1 (antes era NaN)')
print(f'  - REGIMEN_LGCP: flag agregado ({df["REGIMEN_LGCP"].mean()*100:.1f}% bajo nueva ley)')
print(f'\nFeatures Nivel 1: {len(FEATURES_RELATIVAS)}')
print(f'Features Nivel 2: {len(FEATURES_ABSOLUTAS)}')
print(f'Semillas de estabilidad: {SEMILLAS_ESTABILIDAD}')

RESUMEN EJECUTIVO - Experimento 02
Dataset:              3,021,336 ofertas


Segmentos UNSPSC:     57


Clases UNSPSC:        2196
Modelos Nivel 1:      57 (por segmento)
Modelos Nivel 2:      656 (por clase, n >= 500)
Cobertura Nivel 2:    2,862,578 (94.7%)

Umbral (P5 SCORE_MIN): -0.0217
Anomalias P5:          151,067 (5.0%)
Anomalias robustas:    108,231 (3.58%)
Anomalias fragiles:    32,255 (1.07%)
% anomalias adjudicadas (post-hoc): 28.0%

Cambios respecto a exp_01:
  - FUE_ADJUDICADO excluida de features (era feature, ahora solo post-hoc)
  - PRODUCTO_DIFIERE corregido (8 digitos): 0.1% difieren
  - TAMANO_PROVEEDOR: Microemprendedor=1 (antes era NaN)
  - REGIMEN_LGCP: flag agregado (54.9% bajo nueva ley)

Features Nivel 1: 50
Features Nivel 2: 50
Semillas de estabilidad: [42, 123, 456, 789, 2026]
